In [2]:
# ============================================================
# D2 – Stage 4 Validation Branch C – Deterministic Normalisation
# 0. Imports and frozen validation configuration
# ============================================================

from google.colab import files
from pathlib import Path

import hashlib
import json
import math
import re
import unicodedata

import numpy as np
import pandas as pd

DOCUMENT_ID = "D2"
BRANCH_ID = "C"
BRANCH_NAME = "Deterministic normalisation"
PARENT_BRANCH = "B"

EXPECTED_RECORD_COUNT = 22

EXPECTED_FIELDS = [
    "Line Item",
    "Unit",
    "Value 2024",
    "Value 2023"
]

REFERENCE_FIELDS = EXPECTED_FIELDS + ["Source Location"]

# Frozen from D2 Branch A validation / Appendix D logic.
MATCH_KEY_FIELDS = ["Line Item"]

NUMERIC_FIELDS = [
    "Value 2024",
    "Value 2023"
]

TEXT_FIELDS = [
    "Line Item",
    "Unit"
]

ALLOWED_UNITS = {
    "EUR millions",
    "EUR"
}

# Stage 1 contains explicit reported values.
# Monetary items are integer EUR millions and EPS values are reported
# to two decimal places. Exact numerical equality after deterministic
# parsing is therefore retained from Branch A/B validation.
NUMERIC_TOLERANCE = 0.0

OUTPUT_DIR = Path("outputs_D2_validation_branch_C")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Document:", DOCUMENT_ID)
print("Branch:", BRANCH_ID, "-", BRANCH_NAME)
print("Expected reference records:", EXPECTED_RECORD_COUNT)

Document: D2
Branch: C - Deterministic normalisation
Expected reference records: 22


In [3]:
# ------------------------------------------------------------
# 1. Upload validation inputs
# ------------------------------------------------------------
# Required:
#   1) D2_reference_values.csv
#   2) D2_branch_C_parsed_extraction.json
#   3) D2_branch_C_structure_check.json
#   4) D2_branch_C_normalisation_check.json
#
# These artefacts have different roles:
# - Stage 1 reference CSV -> content ground truth
# - parsed extraction     -> preserved Branch C model output used for comparison
# - structure check       -> schema/technical validity
# - normalisation check   -> B→C representation integrity
#
# No Stage 1 values are used to modify or repair the Branch C extraction.

uploaded = files.upload()
uploaded_files = list(uploaded.keys())

csv_files = [f for f in uploaded_files if f.lower().endswith(".csv")]
json_files = [f for f in uploaded_files if f.lower().endswith(".json")]

if len(csv_files) != 1:
    raise ValueError("Upload exactly one Stage 1 reference-values CSV.")

if len(json_files) != 3:
    raise ValueError(
        "Upload exactly three JSON files: Branch C parsed extraction, "
        "structure check, and normalisation check."
    )

REFERENCE_FILE = csv_files[0]

PARSED_EXTRACTION_FILE = None
STRUCTURE_CHECK_FILE = None
NORMALISATION_CHECK_FILE = None

for file_name in json_files:
    with open(file_name, "r", encoding="utf-8") as f:
        obj = json.load(f)

    if (
        isinstance(obj, dict)
        and obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH_ID
        and isinstance(obj.get("records"), list)
    ):
        PARSED_EXTRACTION_FILE = file_name

    if (
        isinstance(obj, dict)
        and obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH_ID
        and "top_level_checks" in obj
        and "record_structure_issues" in obj
        and (
            "schema_validity" in obj
            or "records_with_structure_issues" in obj
        )
    ):
        STRUCTURE_CHECK_FILE = file_name

    if (
        isinstance(obj, dict)
        and obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH_ID
        and obj.get("parent_branch") == PARENT_BRANCH
        and "normalisation_integrity_passed" in obj
        and "parent_equivalence_passed" in obj
    ):
        NORMALISATION_CHECK_FILE = file_name

if PARSED_EXTRACTION_FILE is None:
    raise ValueError("Could not identify D2 Branch C parsed extraction JSON.")

if STRUCTURE_CHECK_FILE is None:
    raise ValueError("Could not identify D2 Branch C structure-check JSON.")

if NORMALISATION_CHECK_FILE is None:
    raise ValueError("Could not identify D2 Branch C normalisation-check JSON.")

print("Reference values:", REFERENCE_FILE)
print("Parsed extraction:", PARSED_EXTRACTION_FILE)
print("Structure check:", STRUCTURE_CHECK_FILE)
print("Normalisation check:", NORMALISATION_CHECK_FILE)

Saving D2_branch_C_parsed_extraction.json to D2_branch_C_parsed_extraction.json
Saving D2_branch_C_normalisation_check.json to D2_branch_C_normalisation_check.json
Saving D2_branch_C_structure_check.json to D2_branch_C_structure_check.json
Saving D2_reference_values.csv to D2_reference_values.csv
Reference values: D2_reference_values.csv
Parsed extraction: D2_branch_C_parsed_extraction.json
Structure check: D2_branch_C_structure_check.json
Normalisation check: D2_branch_C_normalisation_check.json


In [4]:
# ------------------------------------------------------------
# 2. Load inputs, verify identity, and preserve provenance
# ------------------------------------------------------------

with open(PARSED_EXTRACTION_FILE, "r", encoding="utf-8") as f:
    extraction_json = json.load(f)

with open(STRUCTURE_CHECK_FILE, "r", encoding="utf-8") as f:
    structure_check = json.load(f)

with open(NORMALISATION_CHECK_FILE, "r", encoding="utf-8") as f:
    normalisation_check = json.load(f)

df_ref_raw = pd.read_csv(REFERENCE_FILE)
df_ext_raw = pd.DataFrame(extraction_json["records"])

for artefact_name, artefact in {
    "parsed extraction": extraction_json,
    "structure check": structure_check,
    "normalisation check": normalisation_check
}.items():
    if artefact.get("document_id") != DOCUMENT_ID:
        raise ValueError(
            f"Unexpected {artefact_name} document_id: "
            f"{artefact.get('document_id')}"
        )

    if artefact.get("branch") != BRANCH_ID:
        raise ValueError(
            f"Unexpected {artefact_name} branch: "
            f"{artefact.get('branch')}"
        )

if normalisation_check.get("parent_branch") != PARENT_BRANCH:
    raise ValueError(
        f"Unexpected Branch C parent: "
        f"{normalisation_check.get('parent_branch')}"
    )

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

input_provenance = {
    "reference_file": REFERENCE_FILE,
    "reference_sha256": sha256_file(REFERENCE_FILE),
    "parsed_extraction_file": PARSED_EXTRACTION_FILE,
    "parsed_extraction_sha256": sha256_file(PARSED_EXTRACTION_FILE),
    "structure_check_file": STRUCTURE_CHECK_FILE,
    "structure_check_sha256": sha256_file(STRUCTURE_CHECK_FILE),
    "normalisation_check_file": NORMALISATION_CHECK_FILE,
    "normalisation_check_sha256": sha256_file(NORMALISATION_CHECK_FILE)
}

print("Reference shape:", df_ref_raw.shape)
print("Extraction shape:", df_ext_raw.shape)

Reference shape: (22, 5)
Extraction shape: (22, 4)


In [5]:
# ------------------------------------------------------------
# 3. Reuse Branch C schema diagnostics
# ------------------------------------------------------------
# IMPORTANT:
# Record-count agreement is NOT part of schema validity.
# Scope/completeness is assessed later using the fixed reference dataset.

top_level_checks = structure_check.get("top_level_checks", {})

required_top_level_checks = [
    "output_is_json_object",
    "document_id_present",
    "document_id_correct",
    "branch_present",
    "branch_correct",
    "records_present",
    "records_is_list"
]

top_level_valid = all(
    bool(top_level_checks.get(check, False))
    for check in required_top_level_checks
)

# Use the preserved Branch C technical diagnostics.
# This reproduces the same schema principle used in Branch A/B.
schema_validity = bool(
    structure_check.get("json_valid", False)
    and top_level_valid
    and int(structure_check.get("records_with_structure_issues", 0)) == 0
    and int(structure_check.get("records_with_type_issues", 0)) == 0
    and int(structure_check.get("records_with_unexpected_units", 0)) == 0
)

schema_diagnostics = {
    "json_valid": bool(structure_check.get("json_valid", False)),
    "top_level_valid": top_level_valid,
    "records_with_structure_issues":
        int(structure_check.get("records_with_structure_issues", 0)),
    "records_with_type_issues":
        int(structure_check.get("records_with_type_issues", 0)),
    "records_with_unexpected_units":
        int(structure_check.get("records_with_unexpected_units", 0)),
    "duplicate_line_item_count":
        int(structure_check.get("duplicate_line_item_count", 0)),
    "missing_values_by_field":
        structure_check.get("missing_values_by_field", {}),
    "schema_validity": schema_validity
}

print("Schema diagnostics:")
print(json.dumps(schema_diagnostics, indent=2, ensure_ascii=False))

Schema diagnostics:
{
  "json_valid": true,
  "top_level_valid": true,
  "records_with_structure_issues": 0,
  "records_with_type_issues": 0,
  "records_with_unexpected_units": 0,
  "duplicate_line_item_count": 0,
  "missing_values_by_field": {
    "Line Item": 0,
    "Unit": 0,
    "Value 2024": 0,
    "Value 2023": 0
  },
  "schema_validity": true
}


In [6]:
# ------------------------------------------------------------
# 4. Reuse Branch C representation-integrity diagnostics
# ------------------------------------------------------------
# These checks evaluate the deterministic B→C transformation itself.
# They are NOT extraction-accuracy metrics.

representation_integrity = {
    "parent_branch":
        normalisation_check.get("parent_branch"),

    "parent_equivalence_passed":
        bool(normalisation_check.get("parent_equivalence_passed", False)),

    "normalisation_integrity_passed":
        bool(normalisation_check.get("normalisation_integrity_passed", False)),

    "represented_line_item_count":
        int(normalisation_check.get("represented_line_item_count", 0)),

    "record_count_preserved":
        bool(normalisation_check.get("record_count_preserved", False)),

    "observation_identity_and_order_preserved":
        bool(normalisation_check.get(
            "observation_identity_and_order_preserved", False
        )),

    "numeric_tokens_preserved":
        bool(normalisation_check.get("numeric_tokens_preserved", False)),

    "all_expected_content_markers_present":
        bool(normalisation_check.get(
            "all_expected_content_markers_present", False
        )),

    "semantic_label_rewriting_applied":
        bool(normalisation_check.get(
            "semantic_label_rewriting_applied", False
        )),

    "accounting_terminology_harmonisation_applied":
        bool(normalisation_check.get(
            "accounting_terminology_harmonisation_applied", False
        )),

    "manual_correction_applied":
        bool(normalisation_check.get("manual_correction_applied", False)),

    "missing_content_reconstruction_applied":
        bool(normalisation_check.get(
            "missing_content_reconstruction_applied", False
        )),

    "value_modification_applied":
        bool(normalisation_check.get("value_modification_applied", False)),

    "value_rounding_applied":
        bool(normalisation_check.get("value_rounding_applied", False)),

    "derived_calculation_applied":
        bool(normalisation_check.get("derived_calculation_applied", False)),

    "reference_values_used_for_transformation":
        bool(normalisation_check.get(
            "reference_values_used_for_transformation", False
        ))
}

print("Branch C representation integrity:")
print(json.dumps(representation_integrity, indent=2, ensure_ascii=False))

if not representation_integrity["normalisation_integrity_passed"]:
    print(
        "WARNING: Branch C normalisation integrity did not pass. "
        "The extraction result can still be described, but Stage 5 "
        "interpretation must distinguish preprocessing loss from "
        "LLM extraction error."
    )

if representation_integrity["reference_values_used_for_transformation"]:
    raise ValueError(
        "Reference values were reportedly used during Branch C "
        "transformation, which violates the experimental design."
    )

Branch C representation integrity:
{
  "parent_branch": "B",
  "parent_equivalence_passed": true,
  "normalisation_integrity_passed": true,
  "represented_line_item_count": 22,
  "record_count_preserved": true,
  "observation_identity_and_order_preserved": true,
  "numeric_tokens_preserved": true,
  "all_expected_content_markers_present": true,
  "semantic_label_rewriting_applied": false,
  "accounting_terminology_harmonisation_applied": false,
  "manual_correction_applied": false,
  "missing_content_reconstruction_applied": false,
  "value_modification_applied": false,
  "value_rounding_applied": false,
  "derived_calculation_applied": false,
  "reference_values_used_for_transformation": false
}


In [7]:
# ------------------------------------------------------------
# 5. Verify the fixed Stage 1 reference and extraction fields
# ------------------------------------------------------------

missing_reference_fields = [
    field for field in REFERENCE_FIELDS
    if field not in df_ref_raw.columns
]

if missing_reference_fields:
    raise ValueError(
        f"Stage 1 reference dataset is missing fields: "
        f"{missing_reference_fields}"
    )

if len(df_ref_raw) != EXPECTED_RECORD_COUNT:
    raise ValueError(
        f"Unexpected Stage 1 reference count: {len(df_ref_raw)} "
        f"(expected {EXPECTED_RECORD_COUNT})."
    )

missing_extraction_columns = [
    field for field in EXPECTED_FIELDS
    if field not in df_ext_raw.columns
]

# Preserve the raw parsed extraction dataframe.
# A separate comparison copy is used below.
df_ext = df_ext_raw.copy()

# Missing extraction fields are represented as null ONLY in the
# comparison copy so content diagnostics can continue.
# Schema validity remains determined by the structure-check artefact.
for field in missing_extraction_columns:
    df_ext[field] = np.nan

df_ref = df_ref_raw[REFERENCE_FIELDS].copy()
df_ext = df_ext[EXPECTED_FIELDS].copy()

print("Reference records:", len(df_ref))
print("Extracted records:", len(df_ext))
print("Missing extraction columns:", missing_extraction_columns)

Reference records: 22
Extracted records: 22
Missing extraction columns: []


In [8]:
# ------------------------------------------------------------
# 6. Deterministic comparison normalisation
# ------------------------------------------------------------
# FROZEN FROM D2 BRANCH A/B VALIDATION.
#
# This is Stage 4 comparison-only normalisation. It is conceptually
# different from Branch C representation normalisation and never
# modifies the preserved model output.

def normalize_text(value):
    if pd.isna(value):
        return ""

    text = unicodedata.normalize("NFKC", str(value))

    for char in [
        "\u00a0", "\u2000", "\u2001", "\u2002", "\u2003",
        "\u2004", "\u2005", "\u2006", "\u2007", "\u2008",
        "\u2009", "\u200a", "\u202f", "\u205f", "\u3000"
    ]:
        text = text.replace(char, " ")

    text = (
        text
        .replace("’", "'")
        .replace("‘", "'")
        .replace("–", "-")
        .replace("—", "-")
    )

    text = re.sub(r"\s+", " ", text)
    return text.strip().casefold()


def normalize_number(value):
    if pd.isna(value):
        return np.nan

    if isinstance(value, bool):
        return np.nan

    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)

    text = unicodedata.normalize("NFKC", str(value)).strip()

    if text == "":
        return np.nan

    text = text.replace("\u00a0", "").replace(" ", "")

    # Parentheses denote negative financial values in the source.
    if text.startswith("(") and text.endswith(")"):
        text = "-" + text[1:-1]

    # D2 source uses commas as thousands separators.
    text = text.replace(",", "")

    try:
        return float(text)
    except (TypeError, ValueError):
        return np.nan


def numbers_match(reference_value, extracted_value):
    ref_num = normalize_number(reference_value)
    ext_num = normalize_number(extracted_value)

    if pd.isna(ref_num) and pd.isna(ext_num):
        return True

    if pd.isna(ref_num) or pd.isna(ext_num):
        return False

    return math.isclose(
        ref_num,
        ext_num,
        rel_tol=0.0,
        abs_tol=NUMERIC_TOLERANCE
    )

In [9]:
# ------------------------------------------------------------
# 7. Create D2 observation keys and verify uniqueness
# ------------------------------------------------------------
# FROZEN FROM D2 BRANCH A:
# observation identity = normalised Line Item.
#
# Source Location is retained as provenance metadata only and is not
# used for alignment because all D2 observations originate on Page 1.

df_ref["match_key"] = df_ref["Line Item"].apply(normalize_text)
df_ext["match_key"] = df_ext["Line Item"].apply(normalize_text)

reference_duplicate_count = int(
    df_ref["match_key"].duplicated(keep=False).sum()
)

if reference_duplicate_count > 0:
    raise ValueError(
        "The fixed Stage 1 reference contains duplicate normalised "
        "Line Item keys."
    )

# Preserve the first extracted occurrence for deterministic one-to-one
# alignment. Any later extraction with the same identity is an
# unsupported duplicate output.
extraction_duplicate_mask = df_ext["match_key"].duplicated(keep="first")
duplicate_extraction_records = df_ext[extraction_duplicate_mask].copy()
df_ext_unique = df_ext[~extraction_duplicate_mask].copy()

print("Reference duplicate observations:", reference_duplicate_count)
print("Additional extracted duplicate observations:",
      len(duplicate_extraction_records))

Reference duplicate observations: 0
Additional extracted duplicate observations: 0


In [10]:
# ------------------------------------------------------------
# 8. One-to-one record alignment
# ------------------------------------------------------------

df_validation = df_ref.merge(
    df_ext_unique,
    on="match_key",
    how="outer",
    suffixes=("_ref", "_ext"),
    indicator=True,
    validate="one_to_one"
)

print(df_validation["_merge"].value_counts(dropna=False))

_merge
both          22
left_only      0
right_only     0
Name: count, dtype: int64


In [11]:
# ------------------------------------------------------------
# 9. Field-level comparison for aligned observations
# ------------------------------------------------------------

matched_mask = df_validation["_merge"] == "both"

# Text fields
for field in TEXT_FIELDS:
    col = f"{field}_match"
    df_validation[col] = False

    df_validation.loc[matched_mask, col] = (
        df_validation.loc[matched_mask].apply(
            lambda row:
                normalize_text(row[f"{field}_ref"])
                == normalize_text(row[f"{field}_ext"]),
            axis=1
        )
    )

# Numerical fields
for field in NUMERIC_FIELDS:
    col = f"{field}_match"
    df_validation[col] = False

    df_validation.loc[matched_mask, col] = (
        df_validation.loc[matched_mask].apply(
            lambda row: numbers_match(
                row[f"{field}_ref"],
                row[f"{field}_ext"]
            ),
            axis=1
        )
    )

MATCH_COLUMNS = [
    f"{field}_match"
    for field in EXPECTED_FIELDS
]

df_validation["all_fields_match"] = (
    matched_mask
    & df_validation[MATCH_COLUMNS].all(axis=1)
)

In [12]:
# ------------------------------------------------------------
# 10. Classify record outcomes
# ------------------------------------------------------------

def classify_record(row):
    if row["_merge"] == "left_only":
        return "missing"

    if row["_merge"] == "right_only":
        return "hallucinated_unsupported"

    if bool(row["all_fields_match"]):
        return "fully_correct"

    return "discrepant"


df_validation["record_status"] = df_validation.apply(
    classify_record,
    axis=1
)

missing_records = df_validation[
    df_validation["record_status"] == "missing"
].copy()

unsupported_records = df_validation[
    df_validation["record_status"] == "hallucinated_unsupported"
].copy()

discrepant_records = df_validation[
    df_validation["record_status"] == "discrepant"
].copy()

fully_correct_records_df = df_validation[
    df_validation["record_status"] == "fully_correct"
].copy()

duplicate_extraction_records["record_status"] = "hallucinated_duplicate"

print("Missing:", len(missing_records))
print("Unsupported unmatched:", len(unsupported_records))
print("Unsupported duplicate extras:", len(duplicate_extraction_records))
print("Discrepant matched:", len(discrepant_records))
print("Fully correct:", len(fully_correct_records_df))

Missing: 0
Unsupported unmatched: 0
Unsupported duplicate extras: 0
Discrepant matched: 0
Fully correct: 22


In [13]:
# ------------------------------------------------------------
# 11. Calculate common validation metrics
# ------------------------------------------------------------

N_REF = int(len(df_ref))
N_EXT = int(len(df_ext))
N_ALIGNED = int((df_validation["_merge"] == "both").sum())

N_MISSING = int(len(missing_records))
N_UNSUPPORTED_UNMATCHED = int(len(unsupported_records))
N_DUPLICATE_EXTRAS = int(len(duplicate_extraction_records))
N_UNSUPPORTED = N_UNSUPPORTED_UNMATCHED + N_DUPLICATE_EXTRAS

N_DISCREPANT = int(len(discrepant_records))
N_CORRECT = int(len(fully_correct_records_df))

# Completeness is scope recovery, independent of field correctness.
completeness = N_ALIGNED / N_REF if N_REF else 0.0
missing_rate = N_MISSING / N_REF if N_REF else 0.0

# Exact-record metrics:
# a record is a true correct extraction only when every requested field matches.
record_precision = N_CORRECT / N_EXT if N_EXT else 0.0
record_recall = N_CORRECT / N_REF if N_REF else 0.0

record_f1 = (
    2 * record_precision * record_recall
    / (record_precision + record_recall)
    if (record_precision + record_recall) > 0
    else 0.0
)

hallucination_rate = (
    N_UNSUPPORTED / N_EXT
    if N_EXT else 0.0
)

discrepancy_rate = (
    N_DISCREPANT / N_ALIGNED
    if N_ALIGNED else 0.0
)

matched_validation = df_validation[
    df_validation["_merge"] == "both"
].copy()

field_accuracy_among_aligned = {}

for field in EXPECTED_FIELDS:
    col = f"{field}_match"

    field_accuracy_among_aligned[field] = (
        float(matched_validation[col].mean())
        if len(matched_validation)
        else 0.0
    )

# Overall field accuracy uses the fixed reference scope as denominator.
# Missing expected records therefore contribute incorrect field instances.
correct_field_instances = int(
    matched_validation[MATCH_COLUMNS].sum().sum()
)

expected_field_instances = int(
    N_REF * len(EXPECTED_FIELDS)
)

overall_field_accuracy = (
    correct_field_instances / expected_field_instances
    if expected_field_instances
    else 0.0
)

print("Reference records:", N_REF)
print("Extracted records:", N_EXT)
print("Aligned records:", N_ALIGNED)
print("Fully correct:", N_CORRECT)
print("Discrepant:", N_DISCREPANT)
print("Missing:", N_MISSING)
print("Unsupported:", N_UNSUPPORTED)
print("Exact F1:", round(record_f1, 4))
print("Overall field accuracy:", round(overall_field_accuracy, 4))

Reference records: 22
Extracted records: 22
Aligned records: 22
Fully correct: 22
Discrepant: 0
Missing: 0
Unsupported: 0
Exact F1: 1.0
Overall field accuracy: 1.0


In [14]:
# ------------------------------------------------------------
# 12. Field-level error summary
# ------------------------------------------------------------

field_error_summary = []

for field in EXPECTED_FIELDS:
    col = f"{field}_match"

    correct_aligned = int(
        matched_validation[col].sum()
    )

    incorrect_aligned = int(
        len(matched_validation)
        - correct_aligned
    )

    field_error_summary.append({
        "field": field,
        "used_in_matching_key":
            field in MATCH_KEY_FIELDS,

        "aligned_records_evaluated":
            int(len(matched_validation)),

        "correct_values_among_aligned":
            correct_aligned,

        "incorrect_values_among_aligned":
            incorrect_aligned,

        "accuracy_among_aligned":
            round(
                correct_aligned / len(matched_validation),
                4
            )
            if len(matched_validation)
            else 0.0,

        "missing_expected_instances":
            N_MISSING,

        "overall_correct_instances":
            correct_aligned,

        "overall_expected_instances":
            N_REF,

        "overall_field_accuracy":
            round(correct_aligned / N_REF, 4)
            if N_REF
            else 0.0
    })

field_error_summary_df = pd.DataFrame(
    field_error_summary
)

display(field_error_summary_df)

,field,used_in_matching_key,aligned_records_evaluated,correct_values_among_aligned,incorrect_values_among_aligned,accuracy_among_aligned,missing_expected_instances,overall_correct_instances,overall_expected_instances,overall_field_accuracy
0,Line Item,True,22,22,0,1.0,0,22,22,1.0
1,Unit,False,22,22,0,1.0,0,22,22,1.0
2,Value 2024,False,22,22,0,1.0,0,22,22,1.0
3,Value 2023,False,22,22,0,1.0,0,22,22,1.0


In [15]:
# ------------------------------------------------------------
# 13. Build final validation summary
# ------------------------------------------------------------

summary = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH_ID,
    "branch_name": BRANCH_NAME,
    "parent_branch": PARENT_BRANCH,

    "reference_records": N_REF,
    "extracted_records": N_EXT,
    "aligned_records": N_ALIGNED,

    "fully_correct_records": N_CORRECT,
    "discrepant_records": N_DISCREPANT,
    "missing_records": N_MISSING,

    # "Unsupported" is the dissertation-facing record-level term.
    # The hallucination aliases are retained for metric continuity.
    "unsupported_records": N_UNSUPPORTED,
    "hallucinated_records": N_UNSUPPORTED,
    "hallucinated_unmatched_records":
        N_UNSUPPORTED_UNMATCHED,
    "hallucinated_duplicate_records":
        N_DUPLICATE_EXTRAS,

    "completeness":
        round(completeness, 4),

    "missing_rate":
        round(missing_rate, 4),

    "record_precision_exact":
        round(record_precision, 4),

    "record_recall_exact":
        round(record_recall, 4),

    "record_f1_exact":
        round(record_f1, 4),

    "hallucination_rate":
        round(hallucination_rate, 4),

    "discrepancy_rate_among_aligned":
        round(discrepancy_rate, 4),

    "overall_field_accuracy":
        round(overall_field_accuracy, 4),

    "field_accuracy_among_aligned": {
        key: round(value, 4)
        for key, value
        in field_accuracy_among_aligned.items()
    },

    "schema_validity":
        schema_validity,

    "schema_diagnostics":
        schema_diagnostics,

    "branch_C_representation_integrity":
        representation_integrity,

    "matching_key_fields":
        MATCH_KEY_FIELDS,

    "comparison_rules_frozen_from_branch_A":
        True,

    "reference_dataset_branch_independent":
        True,

    "comparison_rules": {
        "record_identity":
            "Normalised Line Item",

        "text":
            "Unicode NFKC, Unicode-space harmonisation, "
            "apostrophe/dash standardisation, whitespace "
            "collapse and case folding",

        "numeric":
            "Exact numerical equality after deterministic parsing",

        "numeric_tolerance":
            NUMERIC_TOLERANCE,

        "source_location_used_for_matching":
            False
    },

    "normalisation_note": (
        "Stage 4 comparison normalisation was applied only to "
        "comparison copies using the rules frozen in D2 Branch A "
        "validation; the preserved Branch C extraction was not modified."
    ),

    "input_provenance":
        input_provenance
}

print(json.dumps(summary, indent=2, ensure_ascii=False))

{
  "document_id": "D2",
  "branch": "C",
  "branch_name": "Deterministic normalisation",
  "parent_branch": "B",
  "reference_records": 22,
  "extracted_records": 22,
  "aligned_records": 22,
  "fully_correct_records": 22,
  "discrepant_records": 0,
  "missing_records": 0,
  "unsupported_records": 0,
  "hallucinated_records": 0,
  "hallucinated_unmatched_records": 0,
  "hallucinated_duplicate_records": 0,
  "completeness": 1.0,
  "missing_rate": 0.0,
  "record_precision_exact": 1.0,
  "record_recall_exact": 1.0,
  "record_f1_exact": 1.0,
  "hallucination_rate": 0.0,
  "discrepancy_rate_among_aligned": 0.0,
  "overall_field_accuracy": 1.0,
  "field_accuracy_among_aligned": {
    "Line Item": 1.0,
    "Unit": 1.0,
    "Value 2024": 1.0,
    "Value 2023": 1.0
  },
  "schema_validity": true,
  "schema_diagnostics": {
    "json_valid": true,
    "top_level_valid": true,
    "records_with_structure_issues": 0,
    "records_with_type_issues": 0,
    "records_with_unexpected_units": 0,
    "d

In [16]:
# ------------------------------------------------------------
# 14. Compact overall-results table
# ------------------------------------------------------------

overall_metrics_df = pd.DataFrame([
    {"metric": "Reference records", "value": N_REF},
    {"metric": "Extracted records", "value": N_EXT},
    {"metric": "Aligned records", "value": N_ALIGNED},
    {"metric": "Fully correct records", "value": N_CORRECT},
    {"metric": "Discrepant records", "value": N_DISCREPANT},
    {"metric": "Missing records", "value": N_MISSING},
    {"metric": "Unsupported records", "value": N_UNSUPPORTED},
    {"metric": "Completeness", "value": round(completeness, 4)},
    {"metric": "Exact precision", "value": round(record_precision, 4)},
    {"metric": "Exact recall", "value": round(record_recall, 4)},
    {"metric": "Exact F1", "value": round(record_f1, 4)},
    {
        "metric": "Overall field accuracy",
        "value": round(overall_field_accuracy, 4)
    },
    {
        "metric": "Hallucination/unsupported rate",
        "value": round(hallucination_rate, 4)
    },
    {"metric": "Schema validity", "value": schema_validity},
    {
        "metric": "Branch C normalisation integrity",
        "value":
            representation_integrity["normalisation_integrity_passed"]
    }
])

display(overall_metrics_df)

,metric,value
0,Reference records,22
1,Extracted records,22
2,Aligned records,22
3,Fully correct records,22
4,Discrepant records,0
5,Missing records,0
6,Unsupported records,0
7,Completeness,1.0
8,Exact precision,1.0
9,Exact recall,1.0


In [17]:
# ------------------------------------------------------------
# 15. Validation integrity checks
# ------------------------------------------------------------

# Every reference observation must be aligned or missing.
assert N_ALIGNED + N_MISSING == N_REF

# Every extracted observation must be aligned, unmatched unsupported,
# or an additional duplicate output.
assert (
    N_ALIGNED
    + N_UNSUPPORTED_UNMATCHED
    + N_DUPLICATE_EXTRAS
    == N_EXT
)

# Every aligned record is either fully correct or discrepant.
assert N_CORRECT + N_DISCREPANT == N_ALIGNED

# Matching identity must be unique in the fixed reference dataset.
assert reference_duplicate_count == 0

for metric_name, metric_value in {
    "completeness": completeness,
    "missing_rate": missing_rate,
    "record_precision": record_precision,
    "record_recall": record_recall,
    "record_f1": record_f1,
    "hallucination_rate": hallucination_rate,
    "discrepancy_rate": discrepancy_rate,
    "overall_field_accuracy": overall_field_accuracy
}.items():
    assert 0.0 <= metric_value <= 1.0, (
        f"Invalid {metric_name}: {metric_value}"
    )

print("Validation integrity checks passed.")

Validation integrity checks passed.


In [18]:
# ------------------------------------------------------------
# 16. Export validation artefacts
# ------------------------------------------------------------

df_validation.to_csv(
    OUTPUT_DIR / "D2_branch_C_validation_detailed.csv",
    index=False
)

missing_records.to_csv(
    OUTPUT_DIR / "D2_branch_C_missing_records.csv",
    index=False
)

unsupported_records.to_csv(
    OUTPUT_DIR / "D2_branch_C_hallucinated_unmatched_records.csv",
    index=False
)

duplicate_extraction_records.to_csv(
    OUTPUT_DIR / "D2_branch_C_hallucinated_duplicate_records.csv",
    index=False
)

discrepant_records.to_csv(
    OUTPUT_DIR / "D2_branch_C_discrepant_records.csv",
    index=False
)

fully_correct_records_df.to_csv(
    OUTPUT_DIR / "D2_branch_C_fully_correct_records.csv",
    index=False
)

field_error_summary_df.to_csv(
    OUTPUT_DIR / "D2_branch_C_field_error_summary.csv",
    index=False
)

overall_metrics_df.to_csv(
    OUTPUT_DIR / "D2_branch_C_overall_metrics.csv",
    index=False
)

with open(
    OUTPUT_DIR / "D2_branch_C_validation_summary.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        summary,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Validation artefacts saved.")

Validation artefacts saved.


In [19]:
# ------------------------------------------------------------
# 17. Final validation report
# ------------------------------------------------------------

final_report = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH_ID,

    "reference_records": N_REF,
    "extracted_records": N_EXT,
    "aligned_records": N_ALIGNED,

    "fully_correct_records": N_CORRECT,
    "discrepant_records": N_DISCREPANT,
    "missing_records": N_MISSING,
    "unsupported_records": N_UNSUPPORTED,

    "completeness":
        round(completeness, 4),

    "record_precision_exact":
        round(record_precision, 4),

    "record_recall_exact":
        round(record_recall, 4),

    "record_f1_exact":
        round(record_f1, 4),

    "overall_field_accuracy":
        round(overall_field_accuracy, 4),

    "schema_validity":
        schema_validity,

    "normalisation_integrity_passed":
        representation_integrity[
            "normalisation_integrity_passed"
        ],

    "comparison_rules_frozen_from_branch_A":
        True
}

print(json.dumps(final_report, indent=2, ensure_ascii=False))

{
  "document_id": "D2",
  "branch": "C",
  "reference_records": 22,
  "extracted_records": 22,
  "aligned_records": 22,
  "fully_correct_records": 22,
  "discrepant_records": 0,
  "missing_records": 0,
  "unsupported_records": 0,
  "completeness": 1.0,
  "record_precision_exact": 1.0,
  "record_recall_exact": 1.0,
  "record_f1_exact": 1.0,
  "overall_field_accuracy": 1.0,
  "schema_validity": true,
  "normalisation_integrity_passed": true,
  "comparison_rules_frozen_from_branch_A": true
}


In [20]:
# ------------------------------------------------------------
# 18. Download validation artefacts
# ------------------------------------------------------------

for output_file in sorted(OUTPUT_DIR.iterdir()):
    if output_file.is_file():
        print("Downloading:", output_file.name)
        files.download(output_file)

Downloading: D2_branch_C_discrepant_records.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloading: D2_branch_C_field_error_summary.csv


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloading: D2_branch_C_validation_summary.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>